In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/healthcare_demo/bronze/raw"

patients = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/patients")
referrals = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/referrals")
appointments = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/appointments")
capacity = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/capacity")

display(patients)
display(referrals)
display(appointments)
display(capacity)

patient_id,age_band,area_code
P0001,18-30,NG1
P0002,46-60,NG2
P0003,31-45,NG2
P0004,18-30,NG1
P0005,61-75,NG1
P0006,18-30,NG1
P0007,31-45,NG2
P0008,18-30,NG2
P0009,61-75,NG2
P0010,61-75,NG5


referral_id,patient_id,referral_date,specialty,priority,referral_status
R0001,P0001,2026-10-23,Dermatology,Routine,Open
R0002,P0002,2026-10-12,Cardiology,Routine,Open
R0003,P0003,2026-07-23,Orthopaedics,Urgent,Open
R0004,P0004,2026-09-11,Orthopaedics,Routine,Open
R0005,P0005,2026-09-23,Orthopaedics,Routine,Open
R0006,P0006,2026-07-28,Orthopaedics,Urgent,Open
R0007,P0007,2026-08-23,Orthopaedics,Urgent,Open
R0008,P0008,2026-09-26,Cardiology,Urgent,Open
R0009,P0009,2026-09-10,Cardiology,Routine,Open
R0010,P0010,2026-08-16,Cardiology,Routine,Open


appointment_id,referral_id,appointment_date,location_id,appointment_status
A0001,R0001,2026-12-27,L002,Attended
A0002,R0002,2026-11-26,L001,Cancelled
A0003,R0003,2026-11-27,L003,Attended
A0004,R0004,2026-12-20,L001,Attended
A0005,R0005,2026-12-08,L001,Cancelled
A0006,R0006,2026-11-28,L001,Attended
A0007,R0007,2026-12-01,L002,Cancelled
A0008,R0008,2026-11-15,L001,Booked
A0009,R0009,2026-11-24,L002,Cancelled
A0010,R0010,2026-11-19,L003,Booked


location_id,location_name,specialty,available_slots
L001,Central Clinic,Cardiology,5
L002,North Clinic,Orthopaedics,8
L003,South Clinic,Dermatology,6


In [0]:
from delta.tables import DeltaTable

def merge_to_delta(source_df, target_table, merge_keys):

    field_names = source_df.schema.fieldNames()
    if "_ingested_at" not in field_names:
        source_df = source_df.withColumns({"_ingested_at": F.current_timestamp()})

    duplicate_keys = (
        source_df.groupBy(merge_keys)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_keys.limit(1).count() > 0:
        raise ValueError(f"Duplicate keys found in {target_table}")

    if not spark.catalog.tableExists(target_table):
        source_df.write.format("delta").saveAsTable(target_table)
        print(f"Created: {target_table}")
    else:
        delta_table = DeltaTable.forName(spark, target_table)

        condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in merge_keys]
        )

        (
            delta_table.alias("target")
            .merge(source_df.alias("source"), condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Merged: {target_table}")

In [0]:
merge_to_delta(patients, "healthcare_demo.bronze.patients", ["patient_id"])
merge_to_delta(referrals, "healthcare_demo.bronze.referrals", ["referral_id"])
merge_to_delta(appointments, "healthcare_demo.bronze.appointments", ["appointment_id"])
merge_to_delta(capacity, "healthcare_demo.bronze.capacity", ["location_id", "specialty"])

Merged: healthcare_demo.bronze.patients
Merged: healthcare_demo.bronze.referrals
Merged: healthcare_demo.bronze.appointments
Merged: healthcare_demo.bronze.capacity
